# Tugas Praktikum – Wisconsin Breast Cancer

Dataset: **Wisconsin Breast Cancer** (569 data, 2 kelas)
- **M** = Malignant (ganas)
- **B** = Benign (jinak)

**Tujuan:** Membangun pipeline klasifikasi menggunakan Logistic Regression dengan SelectKBest untuk seleksi fitur.

## 1. Import Library

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings("ignore")

## 2. Load dan Eksplorasi Data

In [2]:
df = pd.read_csv("dataset/wbc.csv")
print("Shape:", df.shape)
df.head()

Shape: (569, 33)


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [3]:
# Cek tipe data dan nilai hilang
print(df.info())
print("\nJumlah nilai kosong per kolom:")
print(df.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    str    
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             569 non-null

In [4]:
# Distribusi label
print("Distribusi kelas diagnosis:")
print(df["diagnosis"].value_counts())
print("\nProporsi:")
print(df["diagnosis"].value_counts(normalize=True).round(3))

Distribusi kelas diagnosis:
diagnosis
B    357
M    212
Name: count, dtype: int64

Proporsi:
diagnosis
B    0.627
M    0.373
Name: proportion, dtype: float64


## 3. Pemisahan Variabel

In [5]:
# Kolom yang TIDAK digunakan
drop_cols = ["id", "Unnamed: 32"]

# Target / label
target_col = "diagnosis"

# Fitur numerik (semua kolom selain yang di-drop dan target)
feature_cols = [c for c in df.columns if c not in drop_cols + [target_col]]

print(f"Jumlah fitur yang digunakan: {len(feature_cols)}")
print("\nDaftar fitur:")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {col}")

Jumlah fitur yang digunakan: 30

Daftar fitur:
   1. radius_mean
   2. texture_mean
   3. perimeter_mean
   4. area_mean
   5. smoothness_mean
   6. compactness_mean
   7. concavity_mean
   8. concave points_mean
   9. symmetry_mean
  10. fractal_dimension_mean
  11. radius_se
  12. texture_se
  13. perimeter_se
  14. area_se
  15. smoothness_se
  16. compactness_se
  17. concavity_se
  18. concave points_se
  19. symmetry_se
  20. fractal_dimension_se
  21. radius_worst
  22. texture_worst
  23. perimeter_worst
  24. area_worst
  25. smoothness_worst
  26. compactness_worst
  27. concavity_worst
  28. concave points_worst
  29. symmetry_worst
  30. fractal_dimension_worst


## 4. Encoding Kolom `diagnosis`

In [6]:
le = LabelEncoder()
y = le.fit_transform(df[target_col])

print("Mapping encoding:")
for cls, enc in zip(le.classes_, le.transform(le.classes_)):
    print(f"  {cls!r:5s} -> {enc}")

print(f"\nDistribusi y: {np.bincount(y)} (0=Benign, 1=Malignant)")

# Buat matriks fitur X
X = df[feature_cols].copy()
print(f"Shape X: {X.shape}")

Mapping encoding:
  'B'   -> 0
  'M'   -> 1

Distribusi y: [357 212] (0=Benign, 1=Malignant)
Shape X: (569, 30)


## 5. Standardisasi Fitur Numerik



## 6. Seleksi Fitur dengan SelectKBest


In [7]:
# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (455, 30), Test: (114, 30)


In [8]:
# Evaluasi berbagai nilai k dengan cross-validation pada data train
k_values = list(range(1, len(feature_cols) + 1))
cv_scores = []

for k in k_values:
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("selector", SelectKBest(score_func=f_classif, k=k)),
        ("clf", LogisticRegression(max_iter=1000, random_state=42))
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="accuracy")
    cv_scores.append(scores.mean())

# Temukan k terbaik
best_k = k_values[np.argmax(cv_scores)]
best_score = max(cv_scores)

print("CV Accuracy per k:")
print(f"  k  | CV Accuracy")
print("-" * 22)
for k, score in zip(k_values, cv_scores):
    marker = " <- TERBAIK" if k == best_k else ""
    print(f"{k:>4} | {score:>12.4f}{marker}")

print(f"\nJumlah fitur terbaik (k): {best_k}")
print(f"CV Accuracy              : {best_score:.4f}")

CV Accuracy per k:
  k  | CV Accuracy
----------------------
   1 |       0.9077
   2 |       0.9253
   3 |       0.9429
   4 |       0.9385
   5 |       0.9385
   6 |       0.9429
   7 |       0.9429
   8 |       0.9429
   9 |       0.9473
  10 |       0.9473
  11 |       0.9473
  12 |       0.9451
  13 |       0.9363
  14 |       0.9385
  15 |       0.9407
  16 |       0.9670
  17 |       0.9692
  18 |       0.9692
  19 |       0.9736 <- TERBAIK
  20 |       0.9714
  21 |       0.9736
  22 |       0.9736
  23 |       0.9714
  24 |       0.9736
  25 |       0.9736
  26 |       0.9736
  27 |       0.9714
  28 |       0.9714
  29 |       0.9692
  30 |       0.9714

Jumlah fitur terbaik (k): 19
CV Accuracy              : 0.9736


## 7. Pipeline Final dengan k Terbaik

In [9]:
# Buat pipeline final dengan k terbaik
pipe_final = Pipeline([
    ("scaler", StandardScaler()),
    ("selector", SelectKBest(score_func=f_classif, k=best_k)),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])

# Latih pada data train
pipe_final.fit(X_train, y_train)

# Prediksi pada data test
y_pred = pipe_final.predict(X_test)

print("=== Logistic Regression + SelectKBest (k={}) ===".format(best_k))
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["Benign (0)", "Malignant (1)"]))

=== Logistic Regression + SelectKBest (k=19) ===
Accuracy: 0.9825

               precision    recall  f1-score   support

   Benign (0)       0.97      1.00      0.99        72
Malignant (1)       1.00      0.95      0.98        42

     accuracy                           0.98       114
    macro avg       0.99      0.98      0.98       114
 weighted avg       0.98      0.98      0.98       114



## 8. Identifikasi Fitur Terpilih

In [10]:
# Ambil nama fitur, skor ANOVA, dan mask fitur terpilih
selector = pipe_final.named_steps["selector"]
feat_names = np.array(feature_cols)
scores_all = selector.scores_
mask       = selector.get_support()

selected_features = feat_names[mask]
selected_scores   = scores_all[mask]

# Urutkan berdasarkan skor ANOVA (descending)
order = np.argsort(selected_scores)[::-1]
selected_features = selected_features[order]
selected_scores   = selected_scores[order]

print(f"Jumlah fitur terpilih: {len(selected_features)}")
print()
print(f"  No | Nama Fitur                     | F-Score (ANOVA)")
print("-" * 56)
for i, (name, score) in enumerate(zip(selected_features, selected_scores), 1):
    print(f"{i:>4} | {name:<30} | {score:>16.2f}")

Jumlah fitur terpilih: 19

  No | Nama Fitur                     | F-Score (ANOVA)
--------------------------------------------------------
   1 | concave points_worst           |           733.72
   2 | perimeter_worst                |           717.25
   3 | radius_worst                   |           692.86
   4 | concave points_mean            |           684.53
   5 | perimeter_mean                 |           548.41
   6 | area_worst                     |           522.19
   7 | radius_mean                    |           511.27
   8 | area_mean                      |           444.86
   9 | concavity_mean                 |           397.59
  10 | concavity_worst                |           319.51
  11 | compactness_mean               |           263.56
  12 | compactness_worst              |           238.20
  13 | radius_se                      |           205.43
  14 | perimeter_se                   |           193.17
  15 | area_se                        |           180.56
  16 

## 9. Perbandingan Semua Nilai k (Top-5)

In [11]:
# Tampilkan top-5 nilai k berdasarkan CV accuracy
top5_idx = np.argsort(cv_scores)[::-1][:5]
print("Top-5 nilai k berdasarkan CV Accuracy:")
print(f" Rank |    k | CV Accuracy")
print("-" * 28)
for rank, idx in enumerate(top5_idx, 1):
    print(f"{rank:>5} | {k_values[idx]:>4} | {cv_scores[idx]:>12.4f}")

Top-5 nilai k berdasarkan CV Accuracy:
 Rank |    k | CV Accuracy
----------------------------
    1 |   26 |       0.9736
    2 |   25 |       0.9736
    3 |   19 |       0.9736
    4 |   21 |       0.9736
    5 |   22 |       0.9736


## 10. Kesimpulan & Jawaban Pertanyaan

### Ringkasan Alur Pemrosesan:
1. **Pemisahan Variabel:**
   - **Tidak digunakan:** `id` (identifier pasien, tidak memiliki nilai prediktif) dan `Unnamed: 32` (seluruh nilai kosong/NaN).
   - **Target (y):** `diagnosis` (Malignant / Benign).
   - **Fitur (X):** 30 variabel numerik pengukuran karakteristik sel kanker.
2. **Encoding:** Kolom `diagnosis` di-encode dengan `LabelEncoder` (`B` -> `0`, `M` -> `1`).
3. **Standardisasi:** Seluruh 30 fitur numerik distandardisasi menggunakan `StandardScaler` di dalam `Pipeline` agar tidak terjadi kebocoran data (*data leakage*).
4. **Seleksi Fitur:** Menggunakan `SelectKBest` dengan fungsi skor ANOVA F-test (`f_classif`) yang diuji dari k = 1 hingga k = 30 menggunakan **5-Fold Cross-Validation**.
5. **Model Evaluasi:** Model dilatih menggunakan `LogisticRegression(max_iter=1000)` dan menghasilkan akurasi data uji (*test set*) sebesar **0.9825 (98.25%)**.

---

### Jawaban Pertanyaan Evaluasi:

#### 1. Berapa jumlah fitur terbaik yang dapat digunakan?
> **Jumlah fitur terbaik adalah 19 fitur (k = 19).**  
> Rata-rata akurasi 5-fold cross-validation tertinggi mencapai **0.9736 (97.36%)** dan pertama kali dicapai pada k = 19. Mengikuti prinsip *parsimony* (model yang lebih sederhana lebih dipilih untuk mencegah *overfitting* dan efisiensi komputasi), maka 19 fitur adalah pilihan paling optimal.

#### 2. Apa saja fitur tersebut?
Berikut adalah daftar ke-19 fitur terpilih yang diurutkan berdasarkan nilai **F-Score (ANOVA)** dari yang tertinggi:

| No | Nama Fitur | F-Score (ANOVA) | Deskripsi / Makna Klinis |
|:--:|:---|:---:|:---|
| 1 | `concave points_worst` | 733.72 | Titik cekung terburuk/terparah pada kontur sel |
| 2 | `perimeter_worst` | 717.25 | Keliling inti sel terburuk |
| 3 | `radius_worst` | 692.86 | Jari-jari inti sel terburuk |
| 4 | `concave points_mean` | 684.53 | Rata-rata jumlah titik cekung pada kontur sel |
| 5 | `perimeter_mean` | 548.41 | Rata-rata keliling inti sel |
| 6 | `area_worst` | 522.19 | Luas area inti sel terburuk |
| 7 | `radius_mean` | 511.27 | Rata-rata jari-jari inti sel |
| 8 | `area_mean` | 444.86 | Rata-rata luas area inti sel |
| 9 | `concavity_mean` | 397.59 | Rata-rata keparahan bagian cekung kontur sel |
| 10 | `concavity_worst` | 319.51 | Keparahan bagian cekung kontur sel terburuk |
| 11 | `compactness_mean` | 263.56 | Rata-rata kepadatan bentuk sel (perimeter^2 / area - 1.0) |
| 12 | `compactness_worst` | 238.20 | Kepadatan bentuk sel terburuk |
| 13 | `radius_se` | 205.43 | Standard error untuk nilai jari-jari sel |
| 14 | `perimeter_se` | 193.17 | Standard error untuk nilai keliling sel |
| 15 | `area_se` | 180.56 | Standard error untuk nilai luas sel |
| 16 | `texture_worst` | 126.12 | Variasi nilai skala abu-abu terburuk |
| 17 | `smoothness_worst` | 103.72 | Variasi lokal panjang jari-jari sel terburuk |
| 18 | `symmetry_worst` | 100.56 | Simetri terburuk dari sel |
| 19 | `texture_mean` | 93.48 | Rata-rata variasi nilai skala abu-abu |

> **Analisis Tambahan:**  
> Fitur berakhiran `_worst` (kondisi paling parah/ekstrem) dan fitur dimensi fisik sel (`concave points`, `perimeter`, `radius`, `area`) mendominasi urutan teratas, menunjukkan bahwa ukuran sel yang membesar serta batas sel yang berlekuk/cekung merupakan indikator paling kuat untuk mendeteksi kanker payudara ganas (*Malignant*).